# Lasso Regression

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "Student_Performance.csv"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "mitgandhi10/dataset-for-multiple-regression",
    file_path,
)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import ElasticNet

In [6]:
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [19]:
df.describe()

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.992900,69.445700,6.530600,4.583300,55.224800
std,2.589309,17.343152,1.695863,2.867348,19.212558
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,71.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


In [12]:
# scaling and encoding column
scaling = ['Previous Scores']
one_hot_encoding = ['Extracurricular Activities']

feature_cols = (
    scaling 
    + one_hot_encoding 
    + ['Hours Studied', 'Sleep Hours', 'Sample Question Papers Practiced']
)

x = df[feature_cols] # input cols
y = df['Performance Index'] # target col

In [13]:
# train-test-split
x_train, x_test,y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [14]:
# individual pipelines
scale_pipeline = Pipeline(
    steps = [('scale', StandardScaler())]
)

encoding_pipeline = Pipeline(
    steps = [('encode',OneHotEncoder(handle_unknown='ignore'))]
)

In [17]:
# preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('scaled_pipeline', scale_pipeline, scaling),
    ('encoded_pipeline', encoding_pipeline, one_hot_encoding)
])

preprocessor

ColumnTransformer(transformers=[('scaled_pipeline',
                                 Pipeline(steps=[('scale', StandardScaler())]),
                                 ['Previous Scores']),
                                ('encoded_pipeline',
                                 Pipeline(steps=[('encode',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Extracurricular Activities'])])

In [20]:
# Model train
model = Pipeline(
    steps = [('preprocess',preprocessor),
             ('regression',ElasticNet())]
)

# hyperparameter tuning
param_grid = {
    'regression__alpha': np.logspace(-4, 2, 20),
    'regression__l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5,
                 0.6, 0.7, 0.8, 0.9, 1.0],
    'regression__fit_intercept': [True, False],
    'regression__max_iter': [1000, 5000, 10000],
    'regression__tol': [0.0001, 0.001, 0.01]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv = 5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)
print('best model: ',grid.best_estimator_)
print('best parameters: ',grid.best_params_)
print('best score: ',grid.best_score_)

/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.218e+04, tolerance: 2.356e+03
  model = cd_fast.enet_coordinate_descent(
/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.215e+04, tolerance: 2.357e+02
  model = cd_fast.enet_coordinate_descent(
/Users/lucifer/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

best model:  Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('scaled_pipeline',
                                                  Pipeline(steps=[('scale',
                                                                   StandardScaler())]),
                                                  ['Previous Scores']),
                                                 ('encoded_pipeline',
                                                  Pipeline(steps=[('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Extracurricular '
                                                   'Activities'])])),
                ('regression',
                 ElasticNet(alpha=np.float64(0.00020691380811147902),
                            l1_ratio=0.1, tol=0.01))])
best parameters:  {'regression__alpha': np.float64(0.00020691380811147902), 'regression__fit_inter

In [23]:
best_model = grid.best_estimator_
y_Pred = best_model.predict(x_test)
print('intercept: ',best_model.named_steps['regression'].intercept_)
print('coefficient: ',best_model.named_steps['regression'].coef_)
print('r2 score: ',r2_score(y_test, y_Pred))

intercept:  55.45572583272316
coefficient:  [17.5592524  -0.45286609  0.1711223 ]
r2 score:  0.8412924801386126
